# SQL Practice GeneratorClaude-powered SQL practice with PostgreSQL and MySQL sandbox.**Workflow:** pick dialect + question type → generate problem → fill diagnostic form → write SQL → Test / Run / Submit.

In [1]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Setup ──
import os, sys, json, uuid
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd

# Load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv(Path(os.getcwd()).parent / '.env')
except Exception:
    pass

# Reload module imports so edits to .py files take effect on cell re-run
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
for mod in ['sql_practice_utils', 'sandbox']:
    if mod in sys.modules:
        del sys.modules[mod]
import sql_practice_utils as spu
import sandbox as sbx

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
GEN_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'generated_problems')
SOLVED_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'solved')
SESSIONS_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'sessions')
ERRORS_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'errors')
for d in [GEN_DIR, SOLVED_DIR, SESSIONS_DIR, ERRORS_DIR]:
    os.makedirs(d, exist_ok=True)

# Init Claude
claude_ready = spu.init_claude()

# Check sandboxes
pg_ok, pg_msg = sbx.check_postgres()
my_ok, my_msg = sbx.check_mysql()
print(f"PostgreSQL: {pg_msg}")
print(f"MySQL:      {my_msg}")
if not pg_ok and not my_ok:
    print("\nNo sandbox reachable. Run `docker compose up -d` from the project root.")

# Shared state across cells
STATE = {
    'problem': None,
    'hint_index': 0,
    'last_action': None,
}

from IPython.display import Javascript, display

# Globally let textareas keep their native keystrokes by stopping
# JupyterLab's keyboard manager from grabbing them:
# - Shift+Enter: would re-run the cell and wipe widget state.
# - Cmd/Ctrl+Z, Cmd/Ctrl+Shift+Z: textarea native undo/redo would otherwise
#   collide with Jupyter's "undo cell action" binding.
# Tab/Shift+Tab are NOT stopped here because each textarea has its own
# Tab-indent handler that runs at the target level and calls preventDefault
# + stopPropagation itself. Stopping Tab here would kill those handlers.
display(Javascript("""
  document.addEventListener('keydown', function(e) {
    var inField = (e.target.tagName === 'TEXTAREA' || e.target.tagName === 'INPUT');
    if (!inField) return;
    if (e.shiftKey && e.key === 'Enter') {
      e.preventDefault();
      e.stopPropagation();
    }
    var modifier = e.metaKey || e.ctrlKey;
    if (modifier && (e.key === 'z' || e.key === 'Z')) {
      // native textarea undo/redo — keep Jupyter out of it
      e.stopPropagation();
    }
  }, true);
"""))

# Auto-resize textareas to fit their content as the user types — applies to
# .diagnose-textarea and .sql-code-editor textareas. Listens for input events
# and grows the textarea height to match scrollHeight. Mirrors nb02 behavior.
display(Javascript("""
  (function() {
    function autoResize(ta) {
      var scrollTop = window.scrollY;
      ta.style.height = "auto";
      ta.style.height = (ta.scrollHeight + 2) + "px";
      window.scrollTo({ top: scrollTop });
    }
    function attach(ta) {
      if (ta.dataset.autoResizeAttached) return;
      ta.dataset.autoResizeAttached = "1";
      ta.style.overflow = "hidden";
      ta.addEventListener("input", function() { autoResize(ta); });
      setTimeout(function() { autoResize(ta); }, 50);
    }
    function poll() {
      document.querySelectorAll(
        ".diagnose-textarea textarea, .sql-code-editor textarea"
      ).forEach(attach);
    }
    poll();
    setInterval(poll, 1000);
  })();
"""))


Claude ready (claude-sonnet-4-5)
PostgreSQL: Postgres reachable
MySQL:      MySQL reachable


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 1. Pick a problem

Choose dialect, question type, and either generate a new problem or replay one you've solved.

In [2]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Problem Picker ──

dialect_dd = widgets.Dropdown(
    options=[('PostgreSQL', 'postgresql'), ('MySQL', 'mysql')],
    value='postgresql',
    description='Dialect:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

qtype_dd = widgets.Dropdown(
    options=[(v['label'], k) for k, v in spu.QUESTION_TYPES.items()
             if not v.get('hidden_in_picker')],
    value='select_analytical',
    description='Type:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='420px'),
)

subtype_dd = widgets.Dropdown(
    options=[('\U0001F3B2 Random (default generator)', None)],
    value=None,
    description='Subtype:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='520px'),
)

def populate_subtypes(*_):
    """Repopulate the subtype dropdown from spu.SUBTYPES for the selected Type.
    Random = the default generator (picks a subtype at random)."""
    subs = spu.SUBTYPES.get(qtype_dd.value, [])
    subtype_dd.options = [('\U0001F3B2 Random (default generator)', None)] + [(label, key) for key, label in subs]
    subtype_dd.value = None
    # always visible so it's discoverable; types without subtypes show only Random

populate_subtypes()


# ---- Scenario + Difficulty pickers (replaces the old pharmacy_sql Category) ----
# Scenario anchor: Random across all industries by default, or pick a specific
# vertical (matches the dropdown in nb02_analyst_interview_drills.ipynb).
# Difficulty controls the complexity of the generated schema + answer key.
scenario_dd = widgets.Dropdown(
    options=[
        ("🎲 Random (any industry)", "random"),
        ("🎯 My App (BooedUp dating app)", "booedup"),
        ("Consumer Social (dating, social, video, podcast)", "consumer_social"),
        ("Marketplace (rideshare, delivery, listings)", "marketplace"),
        ("Ecommerce (D2C, fashion, grocery)", "ecommerce"),
        ("Fintech (neobank, BNPL, robo, crypto)", "fintech"),
        ("B2B SaaS (CRM, PM, HR, observability)", "b2b_saas"),
        ("Productivity & Media (notes, streaming, news)", "productivity_media"),
        ("Health & Wellness (telehealth, fitness, sleep)", "health_wellness"),
        ("Gaming (mobile, console)", "gaming"),
        ("Education (courses, language, tutoring)", "education"),
        ("Pharmacy & Care (digital pharmacy, diagnostics)", "pharmacy_care"),
    ],
    value="random",
    description="Scenario:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="480px"),
)

difficulty_radio = widgets.RadioButtons(
    options=[("🟢 Easy", "easy"), ("🟡 Moderate", "moderate"), ("🔴 Hard", "hard")],
    value="moderate",
    description="Difficulty:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="420px"),
)


source_radio = widgets.RadioButtons(
    options=[('New (generate)', 'new'), ('Solved (replay)', 'solved')],
    value='new',
    description='Source:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

solved_dd = widgets.Dropdown(
    options=[('— pick a saved problem —', None)],
    description='Saved:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='600px', display='none'),
)

generate_btn = widgets.Button(description='Generate Problem', button_style='primary',
                              layout=widgets.Layout(width='200px', height='34px'))
status_out = widgets.Output()
problem_out = widgets.Output()

def refresh_solved_list(*_):
    items = spu.list_problems(GEN_DIR, dialect=dialect_dd.value, qtype=qtype_dd.value, subtype=subtype_dd.value)
    opts = [('— pick a saved problem —', None)] + [
        (f"{p['generated_at'][:16]} · {p['title']}", p['path']) for p in items
    ]
    solved_dd.options = opts
    solved_dd.value = None

def on_source_change(change):
    if change['new'] == 'solved':
        solved_dd.layout.display = 'flex'
        refresh_solved_list()
    else:
        solved_dd.layout.display = 'none'

source_radio.observe(on_source_change, names='value')
dialect_dd.observe(refresh_solved_list, names='value')
qtype_dd.observe(refresh_solved_list, names='value')
qtype_dd.observe(populate_subtypes, names='value')
subtype_dd.observe(refresh_solved_list, names='value')

def render_problem(p):
    with problem_out:
        clear_output(wait=True)
        title = p.get('title', 'Untitled')
        prompt_html = spu.prompt_to_bullets(p.get('prompt', ''))
        schema_html = spu.schema_to_html(p.get('schema_ddl', ''))
        data_html = spu.insert_data_to_html(p.get('example_input_data', ''))
        ex_cols = p.get('example_output_columns', [])
        ex_rows = p.get('example_output_rows', [])
        ex_df = pd.DataFrame(ex_rows, columns=ex_cols)
        meta = p.get('_meta', {})
        html = f'''
        <div style="border:1px solid #d0d7de; border-radius:6px; padding:16px; background:#fafbfc;">
          <div style="font-size:11px; color:#57606a; margin-bottom:8px;">{meta.get('dialect','')} · {meta.get('question_type','')} · id {meta.get('problem_id','')}</div>
          <h3 style="margin:0 0 12px;">{title}</h3>
          <h4 style="margin-top:6px;">Prompt</h4>
          {prompt_html}
          <h4 style="margin-top:18px;">Schema</h4>
          {schema_html}
          <h4 style="margin-top:18px;">Example input data</h4>
          {data_html}
          <h4 style="margin-top:18px;">Expected output (example data)</h4>
          {ex_df.to_html(index=False, classes='ex-out')}
        </div>
        '''
        display(HTML(html))

def _attempt_progress(attempt, total, last_error):
    with status_out:
        if last_error:
            print(f'Attempt {attempt-1}/{total} failed validation: {last_error[:200]}')
            print(f'Retrying (attempt {attempt}/{total}) ...')
        else:
            print(f'Generating new {qtype_dd.value} problem in {dialect_dd.value} (attempt {attempt}/{total}) ...')

def on_generate(b):
    with status_out:
        clear_output(wait=True)
        if source_radio.value == 'solved':
            path = solved_dd.value
            if not path:
                print('Pick a saved problem first.')
                return
            print('Loading saved problem ...')
            problem = spu.load_problem(path)
        else:
            problem = spu.generate_problem(
                qtype_dd.value, dialect_dd.value,
                on_attempt=_attempt_progress,
                scenario_mode=scenario_dd.value,
                difficulty=difficulty_radio.value,
                subtype=subtype_dd.value,
            )
            if not problem:
                print('Generation failed validation. Click Generate to try again.')
                return
            saved = spu.save_problem(problem, GEN_DIR)
            print(f'Validated and saved: {os.path.basename(saved)}')
            attempts = problem.get('_meta', {}).get('validation_attempts', 1)
            print(f'(passed validation on attempt {attempts})')
        STATE['problem'] = problem
        # Show what was generated (type / subtype / scenario) so you know which pattern
        # to practice; reveals what 'Random' actually landed on.
        _m = problem.get('_meta', {})
        _qt = _m.get('question_type', '?')
        _oqt = _m.get('original_qtype', _qt)
        _eff_sub = (_m.get('subtype') or _m.get('islands_flavor')
                    or _m.get('percentile_flavor') or _m.get('pivot_flavor'))
        _qt_disp = _qt if (_oqt in (None, _qt)) else f'{_oqt} → {_qt}'
        _is_new = (source_radio.value != 'solved')
        _sub_rand = '  (🎲 Random picked this)' if (_is_new and subtype_dd.value is None and _eff_sub) else ''
        _sce_rand = '  (🎲 Random picked this)' if (_is_new and scenario_dd.value == 'random' and _m.get('scenario')) else ''
        print(f'Type:     {_qt_disp}')
        print(f"Subtype:  {_eff_sub if _eff_sub else '(this type has no subtypes)'}{_sub_rand}")
        print(f"Scenario: {_m.get('scenario', '?')}{_sce_rand}")
        STATE['hint_index'] = 0
        # Refresh the problem reminder in the SQL editor section if it's been created
        try:
            refresh_reminder()
        except NameError:
            pass
        # Load schema and example data into the sandbox for the user to start with
        try:
            sbx.reset(dialect_dd.value)
            sbx.execute_script(dialect_dd.value, problem.get('schema_ddl', ''))
            sbx.execute_script(dialect_dd.value, problem.get('example_input_data', ''))
            print('Sandbox loaded with example data.')
        except Exception as e:
            print(f'Sandbox load error: {e}')
        render_problem(problem)

generate_btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HBox([dialect_dd, qtype_dd]),
    subtype_dd,
    scenario_dd,
    difficulty_radio,
    source_radio,
    solved_dd,
    generate_btn,
    status_out,
    problem_out,
]))

## 2. Diagnose before coding

Work the problem through six steps before you touch SQL. Each step is its own accordion panel:

1. **Map the inputs and the output** — input tables and columns, output columns, output grain, row count direction.
2. **Classify the inputs** — single table, join 2+, reshape, union, or procedural.
3. **Name the shape** — say it in one sentence, pick the recipe, list the moves in order.
4. **Walk the decision tree by elimination** — rule out branches, name what survives.
5. **Grab the recipe, adapt from a linked problem** — template, closest prior problem, adaptations.
6. **Sanity check** — failure mode plus a checklist of checks and red flags.

**Solve vs Walkthrough** — Solve mode shows generic guidance in each step hint. Walkthrough mode fetches a worked diagnostic for this specific problem into each hint so you can read it, then paraphrase it back into the fields in your own words.

Click **Get Feedback** to grade the whole worksheet. Skip it entirely if you want to jump straight to coding.


In [3]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ══════════════════════════════════════════════════════════════════════════
# Six-step Diagnostic Worksheet — INTERACTIVE v2
# Steps 3, 4, 5 are driven by checkbox tables (option + lay definition) that
# AUTO-FILL the text fields, which you can then edit or keep:
#   Step 3  tick a recipe        -> "Named shape" template sentence fills in
#           tick composite moves -> "Composite moves" outline fills in
#   Step 4  tick branches to rule out -> "Branches ruled out" fills with the
#           ticked ones; "Remaining branch" fills with the ones you did NOT tick
#   Step 5  tick adaptations     -> "Adaptation notes" fills in
#           click "Load skeleton" -> "Recipe template" fills from the engine's
#           canonical code skeleton for THIS problem's question type
# Every GridBox owns its own scrollbar (overflow 'hidden auto') so a wide table
# never stretches the notebook. Auto-fill is GUARDED: once you hand-edit a
# field, the checkboxes stop overwriting it (clear the field to re-enable).
# ══════════════════════════════════════════════════════════════════════════

_diag_state = {'mode': 'solve', 'walkthrough': None}
# remembers the last value we auto-filled per field, so a hand edit is detected
_auto_last = {}

_LBL = {'description_width': '150px'}
_DDW = widgets.Layout(width='460px')


def _diag_ta(placeholder, desc, min_h='70px'):
    ta = widgets.Textarea(
        placeholder=placeholder, description=desc, style=_LBL,
        layout=widgets.Layout(width='100%', min_height=min_h))
    ta.add_class('diagnose-textarea')
    return ta


def _guarded_fill(ta, fid, new_val):
    """Set ta.value to new_val only if the user has not hand-edited it.
    Empty / placeholder-only fields are fillable; a field still holding the
    last auto value is fillable; anything else is treated as a hand edit and
    left alone."""
    prev = _auto_last.get(fid)
    cur = ta.value
    if cur.strip() == '' or (prev is not None and cur == prev):
        ta.value = new_val
        _auto_last[fid] = new_val


# ---- GridBox checkbox-table builder (nb02 pattern) ----
_GRID_HEADER_STYLE = (
    "padding:8px 10px; background:#f6f8fa; border-bottom:2px solid #2d2d2d; "
    "font-size:11px; text-transform:uppercase; letter-spacing:0.04em; "
    "font-weight:600; color:#1a1a1a; box-sizing:border-box;")
_GRID_CELL_BASE = (
    "padding:6px 10px; border-bottom:1px solid #eee; font-size:12px; "
    "box-sizing:border-box; overflow-wrap:anywhere; word-break:break-word; "
    "line-height:1.5;")


def _make_check_table(specs, headers, col_template, max_height='280px'):
    children = [widgets.HTML(f"<div style='{_GRID_HEADER_STYLE}'>&nbsp;</div>")]
    for h in headers:
        children.append(widgets.HTML(f"<div style='{_GRID_HEADER_STYLE}'>{h}</div>"))
    boxes = {}
    for spec in specs:
        key, cols = spec[0], spec[1:]
        cb = widgets.Checkbox(value=False, indent=False,
                              layout=widgets.Layout(width='auto', min_width='0',
                                                    margin='0', padding='6px 8px 6px 12px'))
        boxes[key] = cb
        children.append(cb)
        for i, txt in enumerate(cols):
            extra = ('font-weight:600; color:#1a1a1a;' if i == 0 else 'color:#444;')
            children.append(widgets.HTML(f"<div style='{_GRID_CELL_BASE} {extra}'>{txt}</div>"))
    grid = widgets.GridBox(children=children, layout=widgets.Layout(
        grid_template_columns=col_template, grid_gap='0', width='100%',
        max_height=max_height, overflow='hidden auto',
        border='1px solid #d0d7de', border_radius='6px', margin='2px 0 0 0'))
    return grid, boxes


def _ticked_keys(boxes):
    return [k for k in boxes if boxes[k].value]


# ---- Mode toggle ----
mode_toggle = widgets.ToggleButtons(
    options=[('Solve — try blind', 'solve'),
             ('Walkthrough — read then paraphrase', 'walkthrough')],
    value='solve', description='Mode:', style={'description_width': '150px'})

# ---- Framing: paraphrase ----
paraphrase_ta = _diag_ta('Restate the prompt in your own words ...',
                         'Paraphrase:', min_h='90px')

# ===== STEP 1 — Map the inputs and the output =====
input_map_ta = _diag_ta('Table:\n    column:\n\nTable:\n    column:',
                        'Input tables/cols:', min_h='110px')
output_columns_ta = _diag_ta('List the columns the output must have ...',
                             'Output columns:')
output_grain_ta = _diag_ta('One row = ... (e.g. one row per customer per month)',
                           'Output grain:')
row_direction_dd = widgets.Dropdown(
    options=[('— pick —', ''), ('Fewer rows', 'fewer'),
             ('Same rows', 'same'), ('Single value', 'single')],
    description='Row count dir:', style=_LBL, layout=_DDW)

# ===== STEP 2 — Classify the inputs =====
input_dd = widgets.Dropdown(
    options=[('— pick —', ''), ('Single table', 'single_table'),
             ('Join 2+', 'join'), ('Reshape', 'reshape'),
             ('Union', 'union'), ('Procedural', 'procedural')],
    description='Data arrives as:', style=_LBL, layout=_DDW)

# ===== Recipe catalog (faithful to the playbook shape-comparison tables) =====
# (recipe_id, family, friendly name, lay definition)
_RECIPE_SPECS = [
    ('row-filter', 'Single table', 'Filter Rows',
     'Output is a subset of rows that pass a check; includes DISTINCT / GROUP BY dedup. The table itself is not changed.'),
    ('group-aggregate', 'Single table', 'Aggregate Per Group',
     'One summary row per group with a count, sum, or average.'),
    ('scalar-extract', 'Single table', 'Extract Single Value',
     'The whole answer is one number or one row, no grouping (MAX, Nth value, overall percentage).'),
    ('rank-partition', 'Single / Multi', 'Rank Within Groups',
     'Top, bottom, or Nth row per group: a window function ranks, then you filter on the rank.'),
    ('row-transform', 'Single table', 'Add or Rewrite Columns',
     'Every row stays but gets a new label or reformatted value that uses only that same row.'),
    ('row-compare', 'Single table', 'Compare Row to Neighbor',
     'Each row looks at the previous or next row to find changes, streaks, or gaps (LAG / LEAD).'),
    ('time-window', 'Single table', 'Rolling or Cumulative Window',
     'Each row gets an aggregate over a date range or a running total up to that point ("rolling", "cumulative", "last N days").'),
    ('normalize-bidirectional', 'Pair', 'Normalize Bidirectional',
     'Rows like (Alice, Bob) and (Bob, Alice) mean the same thing and must be merged before counting.'),
    ('gaps-and-islands', 'Special', 'Gaps and Islands',
     'Collapse consecutive rows into runs (islands) or find the breaks (gaps); usually a row-number difference trick.'),
    ('enrich-join', 'Multi table', 'Look Up Columns',
     'Attach descriptive columns from a reference table to fact rows (includes self-joins); row count stays about the same, no aggregation.'),
    ('enrich-aggregate', 'Multi table', 'Join Then Aggregate',
     'Combine tables, then produce summary stats; output has fewer rows than either input.'),
    ('reshape', 'Reshape', 'Reshape (pivot / unpivot / combine)',
     'Rows to columns, columns to rows, or stack independent queries that share the same columns.'),
    ('function-wrapped', 'Procedure', 'Wrap in a Function',
     'The answer is a reusable function (RETURNS scalar or RETURNS TABLE) called with arguments.'),
    ('do-block-sequential', 'Procedure', 'Sequential DO Block',
     'A procedural block that runs steps in order: bulk UPDATEs or a row-by-row loop.'),
    ('delete-duplicates', 'Cleanup', 'Remove Duplicate Rows',
     'You physically DELETE rows; after the query runs the table itself has changed.'),
]
_RECIPE_FRIENDLY = {k: name for k, fam, name, defn in _RECIPE_SPECS}
_RECIPE_DEFN = {k: defn for k, fam, name, defn in _RECIPE_SPECS}
_RECIPE_LABELS = {k: f'{name} ({k})' for k, fam, name, defn in _RECIPE_SPECS}
_recipe_grid_specs = [(k, fam, name, defn) for k, fam, name, defn in _RECIPE_SPECS]
_RECIPE_COLS = '44px 15% 23% auto'
_RECIPE_HEADERS = ('Family', 'Recipe', 'What it is')

# ===== Composite-moves catalog (building blocks of the query) =====
# (move_id, friendly name, lay definition)
_MOVE_SPECS = [
    ('filter', 'Filter rows', 'Keep only rows that pass a WHERE check.'),
    ('group_agg', 'Group and aggregate', 'GROUP BY a key, compute SUM / COUNT / AVG per group.'),
    ('having', 'Filter on an aggregate', 'HAVING to keep groups whose total / count meets a cutoff.'),
    ('join', 'Join another table', 'JOIN a second table on a key to bring in its columns.'),
    ('left_join_zero', 'Left join + zero-fill', 'LEFT JOIN so unmatched rows stay, then COALESCE missing values to 0.'),
    ('self_join', 'Self-join', 'Join a table to itself to look up or compare within the same table.'),
    ('window_rank', 'Rank with a window', 'ROW_NUMBER / RANK / DENSE_RANK over a PARTITION to rank rows per group.'),
    ('filter_rank', 'Filter on the rank', 'Keep only rows where the rank meets a cutoff (top N, most recent).'),
    ('running', 'Running / cumulative total', 'SUM OVER an ordered window for a running total up to each row.'),
    ('lag_lead', 'Compare to neighbor', 'LAG / LEAD to pull the previous or next row’s value.'),
    ('date_bucket', 'Bucket by date part', 'DATE_TRUNC or EXTRACT to group rows by month, week, or hour.'),
    ('calendar', 'Build a calendar / number spine', 'generate_series to make every date or bucket, then LEFT JOIN actuals.'),
    ('case_label', 'Relabel with CASE', 'CASE WHEN to bucket or rename a value into categories.'),
    ('pivot', 'Pivot long to wide', 'aggregate(CASE WHEN cat THEN val) to turn rows into columns.'),
    ('unpivot', 'Unpivot wide to long', 'UNION ALL of one SELECT per column to turn columns into rows.'),
    ('dedupe', 'Deduplicate', 'DISTINCT, or ROW_NUMBER = 1 per key, to drop repeats.'),
    ('scalar_sub', 'Scalar subquery', 'A subquery that returns one value to compare against.'),
    ('cte', 'CTE staircase', 'Break the query into named CTEs, one verifiable step each.'),
]
_MOVE_FRIENDLY = {k: name for k, name, defn in _MOVE_SPECS}
_move_grid_specs = [(k, name, defn) for k, name, defn in _MOVE_SPECS]

# ===== Adaptation catalog =====
_ADAPT_SPECS = [
    ('join_keys', 'Join keys', 'Which column(s) link the tables; confirm they match in name and type.'),
    ('column_names', 'Column names', 'Map or rename source columns to the exact output names the prompt asks for.'),
    ('filters', 'Filter conditions', 'WHERE / HAVING limits to apply, and where the date or status bounds come from.'),
    ('output_columns', 'Output columns', 'The exact columns, their order, and any rounding the result must have.'),
    ('grain', 'Output grain', 'Confirm one row per what: the level the final SELECT returns.'),
    ('nulls', 'NULL handling', 'What happens to NULLs in keys, filters, and aggregates (NOT IN trap, COALESCE).'),
    ('duplicates', 'Duplicates', 'Whether to dedupe and on which key, or whether duplicates are expected.'),
    ('empty_groups', 'Empty groups / zero-fill', 'Groups with no rows: LEFT JOIN a skeleton and COALESCE to 0 so they still appear.'),
    ('sort_tiebreak', 'Sort and tiebreak', 'Deterministic ORDER BY, with a tiebreaker if you use LIMIT or top-N.'),
]
_ADAPT_FRIENDLY = {k: name for k, name, defn in _ADAPT_SPECS}
_adapt_grid_specs = [(k, name, defn) for k, name, defn in _ADAPT_SPECS]

# ===== STEP 3 — Name the shape (recipe table + moves table, both autofill) =====
named_shape_ta = _diag_ta('Tick a recipe below and this fills in — edit or keep.',
                          'Named shape:')
_recipe_pick_table, _recipe_pick_boxes = _make_check_table(
    _recipe_grid_specs, _RECIPE_HEADERS, _RECIPE_COLS, max_height='300px')
moves_ta = _diag_ta('Tick moves below and this fills in as an outline — reorder freely.',
                    'Composite moves:', min_h='100px')
_moves_table, _moves_boxes = _make_check_table(
    _move_grid_specs, ('Move', 'What it does'), '44px 26% auto', max_height='300px')


def _on_recipe_pick_change(_change):
    picked = _ticked_keys(_recipe_pick_boxes)
    if not picked:
        return
    if len(picked) == 1:
        k = picked[0]
        d = _RECIPE_DEFN[k].rstrip('.')
        d = d[:1].lower() + d[1:] if d else d
        sentence = f'This is a {_RECIPE_FRIENDLY[k]} because {d}.'
    else:
        names = ', '.join(_RECIPE_FRIENDLY[k] for k in picked)
        sentence = (f'This is a composite of: {names}. Because the problem needs '
                    f'each of those moves in sequence.')
    _guarded_fill(named_shape_ta, 'named_shape', sentence)


def _on_moves_change(_change):
    picked = _ticked_keys(_moves_boxes)
    if not picked:
        return
    outline = '\n'.join(f'Move {i+1}: {_MOVE_FRIENDLY[k]}'
                        for i, k in enumerate(picked))
    _guarded_fill(moves_ta, 'moves', outline)


for _cb in _recipe_pick_boxes.values():
    _cb.observe(_on_recipe_pick_change, names='value')
for _cb in _moves_boxes.values():
    _cb.observe(_on_moves_change, names='value')

# ===== STEP 4 — Walk the decision tree by elimination =====
# Tick branches to rule OUT. Ticked -> "Branches ruled out"; unticked ->
# "Remaining branch".
_recipe_ruleout_table, _recipe_ruleout_boxes = _make_check_table(
    _recipe_grid_specs, _RECIPE_HEADERS, _RECIPE_COLS, max_height='300px')
ruled_out_ta = _diag_ta('Tick branches above to rule out — they fill here. Add "because ...".',
                        'Branches ruled out:', min_h='100px')
remaining_ta = _diag_ta('The branches you did NOT rule out fill here automatically.',
                        'Remaining branch:', min_h='90px')


def _on_ruleout_change(_change):
    out_keys = _ticked_keys(_recipe_ruleout_boxes)
    keep_keys = [k for k in _recipe_ruleout_boxes if not _recipe_ruleout_boxes[k].value]
    # only act once at least one box is ticked, so we don't dump all 15 on load
    if not out_keys:
        return
    ruled = '\n'.join(f'Not {_RECIPE_FRIENDLY[k]} because ' for k in out_keys)
    remaining = '\n'.join(_RECIPE_FRIENDLY[k] for k in keep_keys)
    _guarded_fill(ruled_out_ta, 'ruled_out', ruled)
    _guarded_fill(remaining_ta, 'remaining', remaining)


for _cb in _recipe_ruleout_boxes.values():
    _cb.observe(_on_ruleout_change, names='value')

# ===== STEP 5 — Grab the recipe, adapt from a linked problem =====
_adapt_table, _adapt_boxes = _make_check_table(
    _adapt_grid_specs, ('Adaptation', 'What to nail down'), '44px 26% auto',
    max_height='280px')
recipe_template_ta = _diag_ta(
    'Click "Load skeleton for this problem" to fill from the engine — or write it.',
    'Recipe template:', min_h='110px')
similar_problem_ta = widgets.Text(
    placeholder='#NNN or the kind of prior problem it resembles',
    description='Similar problem:', style=_LBL, layout=_DDW)
adaptations_ta = _diag_ta('Tick adaptations above and they fill here — add the specifics.',
                          'Adaptation notes:', min_h='110px')

load_skeleton_btn = widgets.Button(
    description='Load skeleton for this problem', button_style='',
    layout=widgets.Layout(width='280px', height='32px'))
load_skeleton_out = widgets.Output()


def _on_adapt_change(_change):
    picked = _ticked_keys(_adapt_boxes)
    if not picked:
        return
    notes = '\n'.join(f'{_ADAPT_FRIENDLY[k]}: ' for k in picked)
    _guarded_fill(adaptations_ta, 'adaptations', notes)


for _cb in _adapt_boxes.values():
    _cb.observe(_on_adapt_change, names='value')


def on_load_skeleton(b):
    with load_skeleton_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.')
            return
        meta = p.get('_meta', {})
        qtype = meta.get('question_type') or meta.get('original_qtype') or ''
        try:
            skel = spu.get_code_reference(qtype, meta.get('islands_flavor'),
                                          meta.get('percentile_flavor'))
        except Exception as e:
            skel = ''
            print(f'Could not load skeleton: {e}')
        if skel:
            # force-fill the template (this is an explicit user action)
            recipe_template_ta.value = skel
            _auto_last['recipe_template'] = skel
            print(f'Loaded the {qtype} skeleton. Edit it to fit this problem.')
        else:
            print(f'No skeleton on file for question type "{qtype}". Write the '
                  f'template in words yourself.')


load_skeleton_btn.on_click(on_load_skeleton)

# ===== STEP 6 — Sanity check =====
failure_mode_ta = _diag_ta('The most likely way this branch goes wrong ...',
                           'Failure mode:')
_check_specs = [
    ('row_count', 'Row count matches Step 1 expectation'),
    ('null_handling', 'NULL handling covered (JOIN keys, WHERE, aggregates)'),
    ('int_division', 'Integer division cast to NUMERIC'),
    ('group_by', 'GROUP BY includes all non-aggregated SELECT columns'),
    ('order_by', 'ORDER BY is deterministic (tiebreaker if using LIMIT)'),
    ('edge_cases', 'Edge cases: empty table, single row, all NULLs, ties'),
]
_redflag_specs = [
    ('rc_wrong', 'Row count wrong'),
    ('grain_wrong', 'Output grain wrong'),
    ('missing_groups', 'Missing expected groups'),
]
_check_boxes = {}
for _k, _label in _check_specs:
    _check_boxes[_k] = widgets.Checkbox(value=False, description=_label,
                                        indent=False, layout=widgets.Layout(width='100%'))
_redflag_boxes = {}
for _k, _label in _redflag_specs:
    _redflag_boxes[_k] = widgets.Checkbox(value=False, description=_label,
                                          indent=False, layout=widgets.Layout(width='100%'))
_checks_box = widgets.VBox(
    [widgets.HTML("<div style='font-weight:600; font-size:12px; color:#1a7f37; "
                  "margin:4px 0 2px;'>Checks to run</div>")]
    + [_check_boxes[k] for k, _ in _check_specs])
_redflags_box = widgets.VBox(
    [widgets.HTML("<div style='font-weight:600; font-size:12px; color:#cf222e; "
                  "margin:8px 0 2px;'>Red flags (if any, go back to Step 4)</div>")]
    + [_redflag_boxes[k] for k, _ in _redflag_specs])

# ---- Per-step hint widgets ----
_STEP_GENERIC = {
    1: "Name every input table and the columns that matter, the exact output "
       "columns, what one output row represents (the grain), and whether the "
       "result has fewer rows, the same rows, or a single value.",
    2: "Decide how the data arrives: one table, a join across 2+ tables, a "
       "reshape (pivot/unpivot/calendar), a union, or a procedural block.",
    3: "Tick the recipe that fits and the Named shape sentence fills in. Tick "
       "the moves you'll use and the Composite moves outline fills in — reorder "
       "it into the order you'll write the code. Edit either freely.",
    4: "Tick every branch you can rule out. The ticked ones fill Branches ruled "
       "out (add your reasons); the ones you leave unticked fill Remaining "
       "branch. Elimination is faster than guessing the answer.",
    5: "Tick the adaptations you must make (they fill the notes). Click Load "
       "skeleton to pull the canonical code template for this problem, then "
       "edit it. Name the closest prior problem.",
    6: "Name how this branch usually breaks, then tick the checks that apply. "
       "Any red flag means go back to Step 4.",
}


def _make_step_hint(step_no):
    return widgets.HTML(
        f"<div class='diag-step-hint' style='background:#eef5fc; "
        f"border-left:3px solid #0969da; padding:8px 12px; margin-bottom:10px; "
        f"font-size:12.5px; line-height:1.5; border-radius:3px;'>"
        f"<strong>Step {step_no} hint:</strong> {_STEP_GENERIC[step_no]}</div>")


_step_hint = {n: _make_step_hint(n) for n in range(1, 7)}

# ---- Walkthrough fetch button ----
walkthrough_btn = widgets.Button(description='Show worked diagnostic',
                                 button_style='success',
                                 layout=widgets.Layout(width='240px', height='34px'))
walkthrough_out = widgets.Output()


def _set_step_hint(step_no, body_html, walk=False):
    color = '#1a7f37' if walk else '#0969da'
    bg = '#f0f8f0' if walk else '#eef5fc'
    head = (f'Step {step_no} — walkthrough' if walk else f'Step {step_no} hint')
    _step_hint[step_no].value = (
        f"<div class='diag-step-hint' style='background:{bg}; "
        f"border-left:3px solid {color}; padding:8px 12px; margin-bottom:10px; "
        f"font-size:12.5px; line-height:1.55; border-radius:3px;'>"
        f"<strong>{head}:</strong> {body_html}</div>")


def _reset_step_hints_to_generic():
    for n in range(1, 7):
        _set_step_hint(n, _STEP_GENERIC[n], walk=False)


def _apply_walkthrough(w):
    def g(k):
        return (w.get(k, '') or '').replace('\n', '<br>')
    _set_step_hint(1,
        f"<u>Inputs</u>: {g('w_input_map')}<br><u>Output columns</u>: "
        f"{g('w_output_columns')}<br><u>Grain</u>: {g('w_grain')}<br>"
        f"<u>Row direction</u>: {g('w_row_direction')}", walk=True)
    _set_step_hint(2, g('w_input_arrival'), walk=True)
    _set_step_hint(3,
        f"<u>Shape</u>: {g('w_named_shape')}<br><u>Recipe</u>: {g('w_recipe')}"
        f"<br><u>Moves</u>: {g('w_moves')}", walk=True)
    _set_step_hint(4,
        f"<u>Rule out</u>: {g('w_ruled_out')}<br><u>Remaining</u>: "
        f"{g('w_remaining')}", walk=True)
    _set_step_hint(5,
        f"<u>Template</u>: {g('w_recipe_template')}<br><u>Similar</u>: "
        f"{g('w_similar_problem')}<br><u>Adaptations</u>: {g('w_adaptations')}",
        walk=True)
    _set_step_hint(6,
        f"<u>Failure mode</u>: {g('w_failure_mode')}<br><u>Checks</u>: "
        f"{g('w_checks')}", walk=True)


def on_walkthrough(b):
    with walkthrough_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.')
            return
        cached = _diag_state.get('walkthrough')
        if cached and cached.get('_pid') == id(p):
            _apply_walkthrough(cached)
            print('Loaded worked diagnostic (cached). Read it, then paraphrase.')
            return
        print('Asking Claude for a worked diagnostic ...')
        result = spu.walkthrough_diagnostic(p)
        clear_output(wait=True)
        if not result:
            print('Walkthrough generation failed. Try again.')
            return
        result['_pid'] = id(p)
        _diag_state['walkthrough'] = result
        _apply_walkthrough(result)
        print('Worked diagnostic loaded into each step hint. Read it, then '
              'paraphrase into the fields below in your own words.')


walkthrough_btn.on_click(on_walkthrough)
_walkthrough_row = widgets.VBox([walkthrough_btn, walkthrough_out])


def _on_mode_change(change):
    _diag_state['mode'] = change['new']
    if change['new'] == 'solve':
        _reset_step_hints_to_generic()
        _walkthrough_row.layout.display = 'none'
        with walkthrough_out:
            clear_output()
    else:
        _walkthrough_row.layout.display = ''
        with walkthrough_out:
            clear_output()
            print('Click "Show worked diagnostic" to load worked answers for '
                  'this problem into each step hint.')


mode_toggle.observe(_on_mode_change, names='value')
_walkthrough_row.layout.display = 'none'

# ---- Feedback button (grades all six steps) ----
feedback_btn = widgets.Button(description='Get Feedback', button_style='info',
                              layout=widgets.Layout(width='180px', height='34px'))
feedback_out = widgets.Output()


def fmt_check(ok, label, msg):
    badge = ('<span style="color:#1a7f37; font-weight:600;">correct</span>' if ok
             else '<span style="color:#cf222e; font-weight:600;">rethink</span>')
    return f'<li><strong>{label}:</strong> {badge} — {msg}</li>'


def _ticked(boxes, specs):
    labels = [lbl for k, lbl in specs if boxes[k].value]
    return '; '.join(labels) if labels else '(none ticked)'


def on_feedback(b):
    with feedback_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        picked_recipe = _ticked_keys(_recipe_pick_boxes)
        recipe_val = '; '.join(_RECIPE_LABELS[k] for k in picked_recipe)
        answers = {
            'paraphrase': paraphrase_ta.value,
            'input_map': input_map_ta.value,
            'output_columns': output_columns_ta.value,
            'output_grain': output_grain_ta.value,
            'row_direction': row_direction_dd.value,
            'input_arrival': input_dd.value,
            'named_shape': named_shape_ta.value,
            'recipe': recipe_val,
            'composite_moves': moves_ta.value,
            'ruled_out': ruled_out_ta.value,
            'remaining_branch': remaining_ta.value,
            'recipe_template': recipe_template_ta.value,
            'similar_problem': similar_problem_ta.value,
            'adaptations': adaptations_ta.value,
            'failure_mode': failure_mode_ta.value,
            'sanity_checks': _ticked(_check_boxes, _check_specs),
            'red_flags': _ticked(_redflag_boxes, _redflag_specs),
        }
        print('Asking Claude to grade ...')
        result = spu.grade_diagnostic(STATE['problem'], answers)
        clear_output(wait=True)
        if not result:
            print('Grading failed.')
            return
        html = ('<div style="border:1px solid #d0d7de; border-radius:6px; '
                'padding:14px; background:#f6f8fa;">')
        html += '<h4 style="margin:0 0 10px;">Diagnostic Feedback</h4>'
        html += '<ul style="line-height:1.7; margin:0 0 0 18px;">'
        html += f'<li><strong>Paraphrase:</strong> {result.get("paraphrase_feedback","")}</li>'
        html += f'<li><strong>Step 1 — map inputs/output:</strong> {result.get("step1_feedback","")}</li>'
        html += fmt_check(result.get('grain_correct', False), 'Grain', 'output grain')
        html += fmt_check(result.get('row_direction_correct', False), 'Row direction', 'fewer / same / single')
        html += fmt_check(result.get('input_correct', False), 'Step 2 — input', result.get('input_feedback', ''))
        html += fmt_check(result.get('recipe_correct', False), 'Step 3 — recipe', result.get('shape_feedback', ''))
        html += f'<li><strong>Step 3 — moves:</strong> {result.get("moves_feedback","")}</li>'
        html += f'<li><strong>Step 4 — elimination:</strong> {result.get("step4_feedback","")}</li>'
        html += f'<li><strong>Step 5 — adapt:</strong> {result.get("step5_feedback","")}</li>'
        html += f'<li><strong>Step 6 — sanity check:</strong> {result.get("step6_feedback","")}</li>'
        html += '</ul>'
        html += f'<p style="margin:10px 0 0; font-style:italic; color:#57606a;">{result.get("overall","")}</p>'
        html += '</div>'
        display(HTML(html))


feedback_btn.on_click(on_feedback)

# ---- Assemble the six accordion panels ----
def _sub(label, color='#0c447c'):
    return widgets.HTML(f"<div style='font-weight:600; font-size:12px; "
                        f"color:{color}; margin:6px 0 2px;'>{label}</div>")

_panel1 = widgets.VBox([_step_hint[1], input_map_ta, output_columns_ta,
                        output_grain_ta, row_direction_dd])
_panel2 = widgets.VBox([_step_hint[2], input_dd])
_panel3 = widgets.VBox([_step_hint[3], named_shape_ta,
                        _sub('Tick the recipe that fits (fills Named shape):'),
                        _recipe_pick_table,
                        _sub('Tick the moves you’ll use (fills Composite moves outline):'),
                        _moves_table, moves_ta])
_panel4 = widgets.VBox([_step_hint[4],
                        _sub('Tick every branch you rule out (ticked = ruled out, '
                             'unticked = remaining):', '#cf222e'),
                        _recipe_ruleout_table, ruled_out_ta, remaining_ta])
_panel5 = widgets.VBox([_step_hint[5],
                        _sub('Tick the adaptations this problem needs (fills the notes):'),
                        _adapt_table, adaptations_ta,
                        load_skeleton_btn, load_skeleton_out,
                        recipe_template_ta, similar_problem_ta])
_panel6 = widgets.VBox([_step_hint[6], failure_mode_ta, _checks_box, _redflags_box])

diag_accordion = widgets.Accordion(
    children=[_panel1, _panel2, _panel3, _panel4, _panel5, _panel6])
diag_accordion.set_title(0, 'Step 1 — Map the inputs and the output')
diag_accordion.set_title(1, 'Step 2 — Classify the inputs')
diag_accordion.set_title(2, 'Step 3 — Name the shape')
diag_accordion.set_title(3, 'Step 4 — Walk the decision tree by elimination')
diag_accordion.set_title(4, 'Step 5 — Grab the recipe, adapt from a linked problem')
diag_accordion.set_title(5, 'Step 6 — Sanity check')
diag_accordion.selected_index = 0


def refresh_diagnostic_form(*_):
    """New problem loaded: drop cached walkthrough, reset hints, clear skeleton
    output. Keeps user-typed field values."""
    _diag_state['walkthrough'] = None
    if _diag_state.get('mode') == 'solve':
        _reset_step_hints_to_generic()
    for area in [walkthrough_out, load_skeleton_out]:
        with area:
            clear_output()


display(widgets.VBox([
    mode_toggle,
    widgets.HTML("<div style='font-size:12px; color:#57606a; margin:2px 0 8px;'>"
                 "<b>Interactive worksheet v2.</b> Frame the problem, then work the "
                 "six steps — Steps 3-5 have checkbox tables that auto-fill the "
                 "fields. Click Get Feedback to grade the whole worksheet.</div>"),
    paraphrase_ta,
    _walkthrough_row,
    diag_accordion,
    feedback_btn,
    feedback_out,
]))

# --- Diagnostic textareas: dracula theme + Tab/Shift+Tab indent handling ---
display(HTML("""
<style>
.diagnose-textarea { width: 100% !important; }
.diagnose-textarea textarea {
  font: 14px/1.5 ui-monospace, Consolas, Menlo, monospace !important;
  tab-size: 4;
  -moz-tab-size: 4;
  border: 1px solid #44475a !important;
  border-radius: 6px !important;
  outline: none !important;
  padding: 8px 10px !important;
  background: #282a36 !important;
  color: #f8f8f2 !important;
  caret-color: #f8f8f2 !important;
  resize: vertical;
  min-height: 90px;
  width: 100% !important;
  box-sizing: border-box;
}
.diagnose-textarea textarea::placeholder { color: #6272a4 !important; }
.diagnose-textarea textarea::selection { background: #44475a !important; }
.diagnose-textarea textarea:focus { border-color: #bd93f9 !important; }
</style>
"""))

display(Javascript(r"""
(function () {
  function setupOne(ta) {
    if (ta.dataset.diagEnhanced) return;
    ta.dataset.diagEnhanced = '1';
    ta.addEventListener('keydown', function (e) {
      if (e.shiftKey && e.key === 'Enter') {
        e.preventDefault(); e.stopPropagation();
        var s = ta.selectionStart, en = ta.selectionEnd, v = ta.value;
        ta.value = v.slice(0, s) + '\n' + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + 1;
        ta.dispatchEvent(new Event('input', { bubbles: true }));
        return;
      }
      if (e.key !== 'Tab') return;
      e.preventDefault();
      e.stopPropagation();
      var s = ta.selectionStart, en = ta.selectionEnd;
      var v = ta.value;
      var indent = '    ';
      if (s === en && !e.shiftKey) {
        ta.value = v.slice(0, s) + indent + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + indent.length;
      } else {
        var before = v.slice(0, s);
        var lineStart = before.lastIndexOf('\n') + 1;
        var block = v.slice(lineStart, en);
        var lines = block.split('\n');
        var newLines;
        if (e.shiftKey) {
          newLines = lines.map(function (l) {
            if (l.startsWith(indent)) return l.slice(indent.length);
            if (l.startsWith('\t')) return l.slice(1);
            return l;
          });
        } else {
          newLines = lines.map(function (l) { return indent + l; });
        }
        var indented = newLines.join('\n');
        ta.value = v.slice(0, lineStart) + indented + v.slice(en);
        ta.selectionStart = lineStart;
        ta.selectionEnd = lineStart + indented.length;
      }
      ta.dispatchEvent(new Event('input', { bubbles: true }));
    });
  }
  function poll() {
    document.querySelectorAll('.diagnose-textarea textarea').forEach(setupOne);
  }
  poll();
  setInterval(poll, 1000);
})();
"""))


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


<IPython.core.display.Javascript object>

## 3. Write your SQL

- **Test**: run against example data, see output (does not check correctness).
- **Run**: compare your output to the expected output for example data.
- **Submit**: run against hidden test data; passing saves to your solved bank.

In [4]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Code Editor ──

def _render_problem_reminder():
    """Render the case study identically to cell 3's render_problem so the
    user sees the same content in the SQL editor section as in 'Pick a
    problem' above.
    """
    p = STATE.get('problem')
    if not p:
        return widgets.HTML('<div style="color:#57606a; padding:10px;"><i>Generate a problem first.</i></div>')
    title = p.get('title', 'Untitled')
    prompt_html = spu.prompt_to_bullets(p.get('prompt', ''))
    schema_html = spu.schema_to_html(p.get('schema_ddl', ''))
    data_html = spu.insert_data_to_html(p.get('example_input_data', ''))
    ex_cols = p.get('example_output_columns', [])
    ex_rows = p.get('example_output_rows', [])
    ex_df = pd.DataFrame(ex_rows, columns=ex_cols)
    meta = p.get('_meta', {})
    return widgets.HTML(f'''
    <div style="border:1px solid #d0d7de; border-radius:6px; padding:16px; background:#fafbfc; margin-bottom:10px;">
      <div style="font-size:11px; color:#57606a; margin-bottom:8px;">{meta.get('dialect','')} · {meta.get('question_type','')} · id {meta.get('problem_id','')}</div>
      <h3 style="margin:0 0 12px;">{title}</h3>
      <h4 style="margin-top:6px;">Prompt</h4>
      {prompt_html}
      <h4 style="margin-top:18px;">Schema</h4>
      {schema_html}
      <h4 style="margin-top:18px;">Example input data</h4>
      {data_html}
      <h4 style="margin-top:18px;">Expected output (example data)</h4>
      {ex_df.to_html(index=False, classes='ex-out')}
    </div>
    ''')

reminder_box = widgets.VBox([_render_problem_reminder()])

def refresh_reminder():
    reminder_box.children = [_render_problem_reminder()]

code_ta = widgets.Textarea(
    value='/* notes */\n\n',
    placeholder='-- Write your SQL here',
    layout=widgets.Layout(width='100%', min_height='240px'),
)
code_ta.add_class('sql-code-editor')

test_btn = widgets.Button(description='Test', button_style='',
                          layout=widgets.Layout(width='110px'))
run_btn  = widgets.Button(description='Run', button_style='info',
                          layout=widgets.Layout(width='110px'))
submit_btn = widgets.Button(description='Submit', button_style='success',
                            layout=widgets.Layout(width='110px'))
hint_btn = widgets.Button(description='Hint', button_style='warning',
                          layout=widgets.Layout(width='110px'))
format_btn = widgets.Button(description='Format', button_style='',
                            layout=widgets.Layout(width='110px'),
                            tooltip='Re-indent SQL using sqlparse; uppercase keywords, align clauses')
code_ref_btn = widgets.Button(description='Code Reference', button_style='',
                              layout=widgets.Layout(width='150px'),
                              tooltip='Show a generic framework skeleton for this question type (NOT the answer to your specific problem)')
result_out = widgets.Output()
hint_out = widgets.Output()

def on_hint(b):
    with hint_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        idx = STATE.get('hint_index', 0)
        text = spu.get_hint(STATE['problem'], idx)
        STATE['hint_index'] = idx + 1
        total = len(STATE['problem'].get('hints', []))
        display(HTML(f'<div style="background:#fff8c5; border-left:4px solid #d4a72c; padding:10px 14px; border-radius:4px;"><strong>Hint {min(idx+1,total)}/{total}:</strong> {text}</div>'))

hint_btn.on_click(on_hint)

def on_format(b):
    """Re-format SQL with sqlglot (dialect-aware; clean CTE / UNION layout)."""
    try:
        import sqlglot
    except ImportError:
        with hint_out:
            clear_output()
            print('Format requires sqlglot. Install once with: pip install sqlglot')
        return
    raw = code_ta.value or ''
    _d = {'postgresql': 'postgres', 'mysql': 'mysql'}.get(_current_dialect())
    try:
        out = sqlglot.transpile(raw, read=_d, pretty=True)
        code_ta.value = '\n\n'.join(s.rstrip(';') + ';' for s in out if s.strip())
    except Exception as e:
        with hint_out:
            clear_output()
            print(f'Could not format (check the SQL syntax): {e}')
format_btn.on_click(on_format)

def on_code_reference(b):
    """Show a generic SQL framework skeleton for the current question type.
    The skeleton uses placeholder names; it is NOT the answer to this problem.
    """
    with hint_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.')
            return
        meta = p.get('_meta', {})
        qtype = meta.get('question_type', '')
        ref = spu.get_code_reference(
            qtype,
            islands_flavor=meta.get('islands_flavor'),
            percentile_flavor=meta.get('percentile_flavor'),
        )
        flavor_label = ''
        if meta.get('islands_flavor'):
            flavor_label = f" · {meta.get('islands_flavor')}"
        elif meta.get('percentile_flavor'):
            flavor_label = f" · {meta.get('percentile_flavor')}"
        import html as _html
        display(HTML(
            f'<div style="border:1px solid #d0d7de; border-radius:6px; padding:10px 14px; '
            f'background:#f6f8fa; margin-top:6px;">'
            f'<div style="font-weight:600; margin-bottom:6px; font-size:13px; color:#0969da;">'
            f'Code Reference — {qtype}{flavor_label} (generic framework, not the answer)</div>'
            f'<pre style="margin:0; background:#282a36; color:#f8f8f2; padding:10px 12px; '
            f'border-radius:4px; font: 13px/1.5 ui-monospace, Consolas, Menlo, monospace; '
            f'overflow-x:auto; white-space:pre;">{_html.escape(ref)}</pre>'
            f'</div>'
        ))
code_ref_btn.on_click(on_code_reference)

def _current_dialect():
    return STATE.get('problem', {}).get('_meta', {}).get('dialect', 'postgresql')

def _load_example_into_sandbox():
    p = STATE.get('problem')
    d = _current_dialect()
    sbx.reset(d)
    sbx.execute_script(d, p.get('schema_ddl', ''))
    sbx.execute_script(d, p.get('example_input_data', ''))

def _load_test_into_sandbox():
    p = STATE.get('problem')
    d = _current_dialect()
    sbx.reset(d)
    sbx.execute_script(d, p.get('schema_ddl', ''))
    sbx.execute_script(d, p.get('test_data', ''))

def _show_df(df, label='Output'):
    if df is None:
        return
    if df.empty:
        display(HTML(f'<div style="color:#57606a;"><b>{label}:</b> empty result set.</div>'))
    else:
        display(HTML(f'<h4>{label}</h4>' + df.to_html(index=False)))

def on_test(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_example_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        _show_df(df, 'Test output (example data)')

def on_run(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_example_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        expected = spu.expected_to_dataframe(STATE['problem'], 'example')
        ok, msg = sbx.compare_results(df, expected)
        color = '#dcfce7' if ok else '#ffebe9'
        bar   = '#1a7f37' if ok else '#cf222e'
        verdict = 'CORRECT on example data' if ok else 'MISMATCH on example data'
        display(HTML(f'<div style="background:{color}; border-left:4px solid {bar}; padding:10px; margin-bottom:10px;"><b>{verdict}</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{msg}</pre></div>'))
        _show_df(df, 'Your output')
        _show_df(expected, 'Expected output')

def on_submit(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_test_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error on hidden test data:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            try:
                spu.log_error(STATE['problem'], code_ta.value, 'sql_error', ERRORS_DIR, error_message=err)
            except Exception as _e:
                print(f'(could not log error: {_e})')
            return
        expected = spu.expected_to_dataframe(STATE['problem'], 'test')
        ok, msg = sbx.compare_results(df, expected)
        color = '#dcfce7' if ok else '#ffebe9'
        bar   = '#1a7f37' if ok else '#cf222e'
        verdict = 'PASS — saved to solved bank' if ok else 'FAIL on hidden test data'
        if ok:
            try:
                spu.save_solved(STATE['problem'], code_ta.value, SOLVED_DIR)
            except Exception as e:
                msg += f'\n(could not save solved record: {e})'
        display(HTML(f'<div style="background:{color}; border-left:4px solid {bar}; padding:10px; margin-bottom:10px;"><b>{verdict}</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{msg}</pre></div>'))
        if not ok:
            try:
                _yc = list(df.columns) if df is not None else None
                _yr = (df.astype(object).where(df.notna(), None).values.tolist()
                       if df is not None else None)
                spu.log_error(STATE['problem'], code_ta.value, 'wrong_answer', ERRORS_DIR,
                              your_columns=_yc, your_rows=_yr)
            except Exception as _e:
                print(f'(could not log error: {_e})')
            _show_df(df, 'Your output (hidden test data)')
            _show_df(expected, 'Expected output (hidden test data)')

test_btn.on_click(on_test)
run_btn.on_click(on_run)
submit_btn.on_click(on_submit)

display(widgets.VBox([reminder_box, code_ta, widgets.HBox([test_btn, run_btn, submit_btn, hint_btn, format_btn, code_ref_btn]), hint_out, result_out]))

# --- SQL editor: dark theme + line-numbers gutter + Tab handler ---
# Plain ipywidgets Textarea (reliable typing). A JS-injected gutter sits to the
# left and shows line numbers. Tab inserts 4 spaces; Shift+Tab outdents.
# CSS gives the textarea a dracula-style dark background. No CodeMirror — that
# kept fighting JupyterLab's keyboard manager and breaking input.
from IPython.display import Javascript

display(HTML("""
<style>
.sql-editor-container {
  display: flex;
  border: 1px solid #44475a;
  border-radius: 6px;
  overflow: hidden;
  background: #282a36;
  margin-top: 6px;
  width: 100% !important;
  box-sizing: border-box;
}
.sql-editor-container .line-gutter {
  background: #21222c;
  color: #6272a4;
  padding: 8px 10px;
  text-align: right;
  font: 13px/1.5 ui-monospace, Consolas, Menlo, monospace;
  user-select: none;
  white-space: pre;
  min-width: 36px;
  border-right: 1px solid #44475a;
  overflow: hidden;
}
.sql-editor-container .line-gutter-inner { will-change: transform; }
.sql-code-editor { width: 100% !important; }
.sql-code-editor textarea {
  font: 14px/1.5 ui-monospace, Consolas, Menlo, monospace !important;
  tab-size: 4;
  -moz-tab-size: 4;
  border: none !important;
  outline: none !important;
  padding: 8px 10px !important;
  margin: 0 !important;
  background: #282a36 !important;
  color: #f8f8f2 !important;
  caret-color: #f8f8f2 !important;
  flex: 1 1 auto;
  resize: vertical;
  min-height: 280px;
  width: 100% !important;
}
.sql-code-editor textarea::placeholder { color: #6272a4 !important; }
.sql-code-editor textarea::selection { background: #44475a !important; }
</style>
"""))

display(Javascript(r"""
(function () {
  function setupOne(ta) {
    if (ta.dataset.sqlEnhanced) return;
    ta.dataset.sqlEnhanced = '1';
    var parent = ta.parentNode;
    var wrap = document.createElement('div');
    wrap.className = 'sql-editor-container';
    var gutter = document.createElement('div');
    gutter.className = 'line-gutter';
    var gutterInner = document.createElement('div');
    gutterInner.className = 'line-gutter-inner';
    gutter.appendChild(gutterInner);
    parent.insertBefore(wrap, ta);
    wrap.appendChild(gutter);
    wrap.appendChild(ta);

    function refresh() {
      var n = (ta.value.match(/\n/g) || []).length + 1;
      var s = '';
      for (var i = 1; i <= n; i++) s += i + '\n';
      gutterInner.textContent = s;
    }
    refresh();
    ta.addEventListener('input', refresh);
    ta.addEventListener('scroll', function () {
      gutterInner.style.transform = 'translateY(' + (-ta.scrollTop) + 'px)';
    });

    ta.addEventListener('keydown', function (e) {
      if (e.shiftKey && e.key === 'Enter') {
        e.preventDefault(); e.stopPropagation();
        var s = ta.selectionStart, en = ta.selectionEnd, v = ta.value;
        ta.value = v.slice(0, s) + '\n' + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + 1;
        ta.dispatchEvent(new Event('input', { bubbles: true }));
        return;
      }
      if (e.key !== 'Tab') return;
      e.preventDefault();
      e.stopPropagation();
      var s = ta.selectionStart, en = ta.selectionEnd;
      var v = ta.value;
      var indent = '    ';
      if (s === en && !e.shiftKey) {
        ta.value = v.slice(0, s) + indent + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + indent.length;
      } else {
        var before = v.slice(0, s);
        var lineStart = before.lastIndexOf('\n') + 1;
        var block = v.slice(lineStart, en);
        var lines = block.split('\n');
        var newLines;
        if (e.shiftKey) {
          newLines = lines.map(function (l) {
            if (l.startsWith(indent)) return l.slice(indent.length);
            if (l.startsWith('\t')) return l.slice(1);
            return l;
          });
        } else {
          newLines = lines.map(function (l) { return indent + l; });
        }
        var indented = newLines.join('\n');
        ta.value = v.slice(0, lineStart) + indented + v.slice(en);
        ta.selectionStart = lineStart;
        ta.selectionEnd = lineStart + indented.length;
      }
      ta.dispatchEvent(new Event('input', { bubbles: true }));
      refresh();
    });
  }
  function poll() {
    document.querySelectorAll('.sql-code-editor textarea').forEach(setupOne);
  }
  poll();
  setInterval(poll, 1000);
})();
"""))


<IPython.core.display.Javascript object>

## 4. Next problem

Clear all fields and start over.

In [5]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Next Question ──
next_btn = widgets.Button(description='Next Question (clear all)', button_style='danger',
                          layout=widgets.Layout(width='240px', height='34px'))
next_out = widgets.Output()

def on_next(b):
    STATE['problem'] = None
    STATE['hint_index'] = 0
    code_ta.value = '/* notes */\n\n'
    # framing + step fields
    paraphrase_ta.value = ''
    input_map_ta.value = ''
    output_columns_ta.value = ''
    output_grain_ta.value = ''
    row_direction_dd.value = ''
    input_dd.value = ''
    named_shape_ta.value = ''
    moves_ta.value = ''
    ruled_out_ta.value = ''
    remaining_ta.value = ''
    recipe_template_ta.value = ''
    similar_problem_ta.value = ''
    adaptations_ta.value = ''
    failure_mode_ta.value = ''
    _auto_last.clear()
    for _cb in _recipe_pick_boxes.values():
        _cb.value = False
    for _cb in _moves_boxes.values():
        _cb.value = False
    for _cb in _recipe_ruleout_boxes.values():
        _cb.value = False
    for _cb in _adapt_boxes.values():
        _cb.value = False
    for _cb in _check_boxes.values():
        _cb.value = False
    for _cb in _redflag_boxes.values():
        _cb.value = False
    # drop cached walkthrough + reset step hints to generic
    try:
        refresh_diagnostic_form()
    except NameError:
        pass
    try:
        refresh_reminder()
    except NameError:
        pass
    for area in [problem_out, feedback_out, result_out, hint_out, status_out, next_out, walkthrough_out, load_skeleton_out]:
        try:
            with area:
                clear_output(wait=True)
        except Exception:
            pass
    with next_out:
        print('Cleared. Generate a new problem above.')

next_btn.on_click(on_next)
display(widgets.VBox([next_btn, next_out]))

---**Files written each run:**- `data/outputs/generated_problems/` — every generated problem (loadable via Source: Solved).- `data/outputs/solved/` — successful submissions with your solution code.**Reset the sandbox** when something gets weird: `docker compose down -v && docker compose up -d`.